# Clase 036 — Plots básicos

**Parte 0** · VanderPlas cap. 4 §§ 4.2-4.5.

> 🎯 5 plots que cubren 80% del EDA. Saber cuándo cada uno.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

# Penguins-like sintético
df = pd.DataFrame({
    'species'    : np.repeat(['Adelie', 'Chinstrap', 'Gentoo'], [50, 30, 40]),
    'body_mass'  : np.concatenate([rng.normal(3700, 400, 50), rng.normal(3700, 400, 30), rng.normal(5050, 500, 40)]),
    'bill_length': np.concatenate([rng.normal(39, 2, 50),     rng.normal(48, 3, 30),     rng.normal(48, 3, 40)]),
    'flipper'    : np.concatenate([rng.normal(190, 6, 50),    rng.normal(196, 7, 30),    rng.normal(217, 7, 40)]),
})

## 1️⃣ Line — tendencias temporales

Úsalo cuando el eje X tiene **orden natural** (tiempo, espacio). NO uses line para variables categóricas — engaña al ojo.

In [ ]:
fechas = pd.date_range('2024-01-01', periods=24, freq='ME')
ventas = (rng.normal(0, 0.5, 24).cumsum() + 10) * 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fechas, ventas, marker='o', linewidth=2, color='steelblue')
max_idx = ventas.argmax()
ax.annotate(f'pico: {ventas[max_idx]:.0f}',
            xy=(fechas[max_idx], ventas[max_idx]),
            xytext=(fechas[max_idx], ventas[max_idx] + 100),
            arrowprops=dict(arrowstyle='->', color='red'),
            ha='center')
ax.set_title('Ventas mensuales 2024')
ax.set_ylabel('USD')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2️⃣ Scatter — relación entre dos continuas

Con `c=` puedes codificar una 3ª dimensión (color), y con `s=` una 4ª (tamaño). Cuidado: más de 3 dimensiones en un scatter sobrecarga.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Adelie': 'C0', 'Chinstrap': 'C1', 'Gentoo': 'C2'}
for sp, sub in df.groupby('species'):
    ax.scatter(sub['bill_length'], sub['body_mass'],
               s=sub['flipper']*1.5, alpha=0.6, label=sp,
               color=colors[sp], edgecolors='white', linewidth=0.5)
ax.set_xlabel('bill_length (mm)')
ax.set_ylabel('body_mass (g)')
ax.set_title('body_mass vs bill_length (tamaño ∝ flipper)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3️⃣ Bar — categóricas

Vertical vs horizontal:
- **Vertical**: si las etiquetas son cortas.
- **Horizontal**: si hay muchas categorías o nombres largos (no tienes que rotar texto).

In [ ]:
counts = df['species'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(counts.index, counts.values, color=['C0','C1','C2'])
ax1.set_title('bar vertical')
ax1.set_ylabel('count')

ax2.barh(counts.index[::-1], counts.values[::-1], color=['C2','C1','C0'])
ax2.set_title('bar horizontal (≈ misma info, mejor con nombres largos)')

plt.tight_layout()
plt.show()

## 4️⃣ Histogram — distribución

**Bins** son críticos:
- Pocos → escondes estructura.
- Muchos → ruido visual.
- `bins='auto'` usa Freedman-Diaconis, buen default.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].hist(df['body_mass'], bins=5,     edgecolor='white')
axes[0].set_title('bins=5 (pocos, esconde)')

axes[1].hist(df['body_mass'], bins='auto', edgecolor='white')
axes[1].set_title("bins='auto' (Freedman-Diaconis)")

axes[2].hist(df['body_mass'], bins=50,    edgecolor='white')
axes[2].set_title('bins=50 (muchos, ruidoso)')

for a in axes: a.set_xlabel('body_mass')
plt.tight_layout()
plt.show()

## 5️⃣ Boxplot — distribución resumida

```
           ┌─── max (whisker)
           │
           │   ┌── Q3 (75%)
           │  ╶┤
           │   │   ── mediana (50%)
           │  ╶┤
           │   └── Q1 (25%)
           │
           └─── min (whisker)
  ° outliers (fuera de whisker = > 1.5×IQR)
```

Útil para comparar **muchos grupos** rápido.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
datos_por_sp = [sub['body_mass'].values for _, sub in df.groupby('species')]
ax.boxplot(datos_por_sp, labels=df['species'].unique().tolist(), patch_artist=True)
ax.set_ylabel('body_mass (g)')
ax.set_title('body_mass por species')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6️⃣ Bar con errorbars + fill_between para bandas

In [ ]:
medias = df.groupby('species')['body_mass'].mean()
stds   = df.groupby('species')['body_mass'].std()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(medias.index, medias.values, yerr=stds.values, capsize=8, color=['C0','C1','C2'], alpha=0.7)
ax.set_ylabel('body_mass (g)')
ax.set_title('Media ± std por species')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## ✅ Checklist

- [ ] Elijo line solo para ejes X con orden natural
- [ ] Uso scatter + c/s para codificar 3-4 dimensiones
- [ ] Pongo bins='auto' por default en histogramas
- [ ] Interpreto las 5 partes de un boxplot
- [ ] Añado errorbars cuando muestro promedios

## 📝 Homework

Ver `README.md`. 5 plots básicos sobre penguins, scatter 3D, bar con errorbars, boxplot agrupado.

## 📖 Definiciones y características

**Line plot**

Une puntos con líneas. Implica continuidad/orden en X — solo úsalo cuando X tiene **orden natural** (tiempo, espacio, secuencia).

**Scatter**

Puntos no conectados. Muestra **relación** entre 2 continuas. Con `c=` codificas 3ª dim (color), con `s=` 4ª (tamaño). Más de 4 dims sobrecarga.

**Bar / barh**

Barras para **categóricas**. Vertical (`bar`) si etiquetas son cortas, horizontal (`barh`) si son largas o muchas.

**Histogram**

Distribución de UNA continua. Bins importan: pocos esconden estructura, muchos generan ruido. `bins='auto'` usa Freedman-Diaconis (buen default).

**Boxplot**

Resumen de distribución: mediana (línea), Q1-Q3 (caja), whiskers (1.5×IQR), outliers (puntos). Útil para comparar muchos grupos rápido.

**Errorbar**

Barra + línea vertical/horizontal indicando incertidumbre (std, IC95%). Sin esto, las barras mienten visualmente.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Line plot entre 2 categorías "A", "B" | Conectar categorías con línea **engaña** (sugiere continuidad inexistente). **Fix**: usa `bar` o `scatter`. |
| Histograma con escala Y rara | Por default `density=False` (counts). Si quieres comparar distribuciones de tamaños distintos, `density=True` (probabilidad). |
| Boxplot todos iguales por outliers extremos | Outliers dominan visualmente; cajas quedan apretadas. **Fix**: `ax.set_ylim(p5, p95)` recorta vista, o reporta los outliers aparte. |
| Bar chart con colores random distrae | Sin codificación significativa, color = ruido. **Fix**: un color único para todas; reserva color para grupos reales. |
| Scatter con miles de puntos = blob negro | Overplotting. **Fix**: `alpha=0.3`, hexbin (`plt.hexbin`), o agrupar por bin antes. |

## ❓ Preguntas frecuentes

**❓ ¿Pie chart cuándo?**

**Casi nunca.** El ojo humano compara mal ángulos. Para proporciones: bar o stacked bar. Pie tolerable solo con 2-3 categorías y proporciones muy distintas.

**❓ ¿Cuántos bins en un histograma?**

`bins='auto'` (Freedman-Diaconis) es buen default. Si es estudios académicos: regla de Sturges (`bins=int(np.log2(n)+1)`). Experimenta con 10/30/50 si dudas.

**❓ ¿Boxplot o violinplot?**

**Boxplot**: rápido, 5 estadísticos, outliers claros. **Violinplot**: muestra distribución completa (multimodalidad). Para comparar 3-10 grupos, ambos OK. Para >10, boxplot gana en densidad.

**❓ ¿Errorbars con std o con IC?**

**Std**: dispersión natural de los datos. **IC95% de la media**: incertidumbre del estimador (más pequeño con N grande). Para inferencia, IC. Para describir, std.

**❓ ¿Plot 3D buen idea?**

**Casi nunca.** Oclusión + perspectiva engañan. 2D con color/tamaño suele comunicar mejor. Excepción: superficies analíticas `z = f(x, y)`.

## 🔗 Referencias

- VanderPlas cap. 4 §§ 4.2-4.5
- [matplotlib gallery](https://matplotlib.org/stable/gallery/index.html)

➡️ **Siguiente:** [037 — subplots y gridspec](../037-matplotlib-subplots-y-gridspec/README.md)

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables sin internet** de los ejercicios del README. Cada bloque incluye comentarios y comprobaciones (`assert`/`print`). Intenta resolverlos tú antes de mirar.

In [ ]:
# --- Datos sintéticos reproducibles tipo "penguins" (sin internet) ---
import numpy as np, pandas as pd
def make_penguins(seed=7):
    rng = np.random.default_rng(seed)
    specs = [("Adelie",152,(3700,190,39,18)),
             ("Chinstrap",68,(3733,196,49,18)),
             ("Gentoo",124,(5076,217,47,15))]
    parts = []
    for name, n, (bm, fl, bl, bd) in specs:
        parts.append(pd.DataFrame({
            "species": name,
            "body_mass_g":       rng.normal(bm, 450, n).round(0),
            "flipper_length_mm": rng.normal(fl, 7,   n).round(1),
            "bill_length_mm":    rng.normal(bl, 3,   n).round(1),
            "bill_depth_mm":     rng.normal(bd, 1.5, n).round(1),
            "sex":               rng.choice(["Male","Female"], n),
        }))
    return pd.concat(parts, ignore_index=True)
penguins = make_penguins()
print("penguins:", penguins.shape, "| especies:", penguins.species.unique().tolist())

**Ejercicio 1 — Line.** Serie temporal de ventas (sintética); anota el máximo con flecha.

In [ ]:
meses = np.arange(1, 13)
rng = np.random.default_rng(1)
ventas = (100 + meses*5 + rng.normal(0, 8, 12)).round(1)
imax = int(np.argmax(ventas))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(meses, ventas, marker='o')
ax.annotate(f'máx = {ventas[imax]}', xy=(meses[imax], ventas[imax]),
            xytext=(meses[imax]-3, ventas[imax]+4),
            arrowprops=dict(arrowstyle='->', color='crimson'))
ax.set_xlabel('mes'); ax.set_ylabel('ventas'); ax.set_title('Ventas mensuales')
print('Mes con máximo:', meses[imax], '->', ventas[imax])
plt.close(fig)

**Ejercicio 2 — Scatter.** `body_mass` vs `bill_length`, color por especie y tamaño por `flipper`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'Adelie': 'tab:blue', 'Chinstrap': 'tab:orange', 'Gentoo': 'tab:green'}
for sp, g in penguins.groupby('species'):
    ax.scatter(g.bill_length_mm, g.body_mass_g,
               s=(g.flipper_length_mm - 170), alpha=.6, label=sp, color=colors[sp])
ax.set_xlabel('bill_length_mm'); ax.set_ylabel('body_mass_g'); ax.legend()
assert len(ax.collections) == 3
print('3 especies ploteadas; el tamaño del punto ~ flipper_length_mm')
plt.close(fig)

**Ejercicio 3 — Bar.** Conteo por especie (desc), vertical y horizontal.

In [ ]:
counts = penguins.species.value_counts().sort_values(ascending=False)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.bar(counts.index, counts.values);           a1.set_title('Vertical')
a2.barh(counts.index[::-1], counts.values[::-1]); a2.set_title('Horizontal')
print(counts.to_string())
print('La horizontal gana legibilidad cuando las etiquetas de categoría son largas.')
plt.close(fig)

**Ejercicio 4 — Histogram.** `body_mass` con `bins='auto'` vs `bins=10`.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
a1.hist(penguins.body_mass_g, bins='auto'); a1.set_title("bins='auto'")
a2.hist(penguins.body_mass_g, bins=10);     a2.set_title('bins=10')
n_auto = len(a1.patches)
print(f"'auto' generó {n_auto} barras; con bins=10 hay 10. "
      "Menos bins -> más suave; más bins -> más detalle (y ruido).")
plt.close(fig)

**Ejercicio 5 — Boxplot.** `body_mass` por especie: 3 cajas e identifica outliers.

In [ ]:
groups = [g.body_mass_g.values for _, g in penguins.groupby('species')]
labels = list(penguins.groupby('species').groups.keys())
fig, ax = plt.subplots(figsize=(7, 5))
bp = ax.boxplot(groups)
ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)
ax.set_ylabel('body_mass_g')
n_out = sum(len(f.get_ydata()) for f in bp['fliers'])
print('Puntos fuera de 1.5*IQR (outliers marcados):', n_out)
plt.close(fig)